In [5]:
import keras
from hgq.utils.sugar import Dataset

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_test = X_test.astype("float32") / 255
y_test = keras.utils.to_categorical(y_test, 10)
dataset_test = Dataset(X_test, y_test, batch_size=2048, device='gpu:0')

In [6]:
from keras.models import load_model

model = load_model('/home/ncgadmin/DAT255/DAT255-project/MNIST/MNIST_CNN_HGQ_DynamicTraining/MNIST_CNN_HGQ_DynamicTraining_model.keras')
score = model.evaluate(dataset_test)

2026-04-27 12:35:19.809353: I external/local_xla/xla/service/service.cc:163] XLA service 0x73a12c031e70 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-04-27 12:35:19.809378: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
2026-04-27 12:35:19.878440: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777286120.235664  157795 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-04-27 12:35:20.239017: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 177271888 exceeds 10% of free system memory.
2026-04-27 12:35:20.338578: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 177271888 exceeds 10% of free system memory.
2026-04-27 12:35:20.402921: W exter

5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 313ms/step - accuracy: 0.9282 - loss: 0.2399                     


2026-04-27 12:35:20.683217: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 156504208 exceeds 10% of free system memory.


In [ ]:
import hls4ml
from hgq.utils import trace_minmax

trace_minmax(model, X_test, verbose=True)

hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')

hls_config['Model']['ReuseFactor'] = 1
hls_config['Model']['Strategy'] = 'Distributed Arithmetic'
proj_name = f"mnist_cnn_hgq_dynamictraining_hls4ml"

hls_model = hls4ml.converters.convert_from_keras_model(
    model,    
    backend   ='vitisunified',
    hls_config= hls_config,
    io_type   ='io_stream',
    proj_name = proj_name,
    board     ='kv260',
    part='xck26-sfvc784-2LV-c',
    clock_period='5'
)
hls_model.compile()

In [ ]:
hls_model.build(csim=False, synth=True, cosim=False)